# Jailbreak Analysis — Interactive Exploration

This notebook provides an interactive interface to explore the jailbreak results.
Run the pipeline first:
```bash
python src/runner.py --mode baseline --limit 20
python src/judge.py --input data/results/baseline.jsonl
```

In [ ]:
import sys
sys.path.insert(0, '..')  # add project root to path

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import BASELINE_LABELED, DEFENSE_FILES, FIGURES_DIR

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Setup complete.')

## 1. Load Results

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)

if BASELINE_LABELED.exists():
    df = load_jsonl(BASELINE_LABELED)
    print(f'Loaded {len(df)} baseline records.')
    df.head(3)
else:
    print(f'File not found: {BASELINE_LABELED}')
    print('Run the baseline pipeline first.')
    df = pd.DataFrame()

## 2. Overall Attack Success Rate

In [ ]:
if not df.empty:
    overall_asr = df['complied'].astype(bool).mean()
    n_complied = df['complied'].astype(bool).sum()
    print(f'Total prompts    : {len(df)}')
    print(f'Complied (YES)   : {n_complied}')
    print(f'Overall ASR      : {overall_asr:.1%}')

## 3. ASR by Category

In [ ]:
if not df.empty:
    by_cat = df.groupby('category')['complied'].apply(lambda s: s.astype(bool).mean()).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(9, max(4, len(by_cat) * 0.5)))
    by_cat.mul(100).plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('Attack Success Rate (%)')
    ax.set_title('ASR by Category')
    plt.tight_layout()
    plt.show()
    
    display(by_cat.mul(100).round(1).rename('ASR (%)').to_frame())

## 4. ASR by Attack Type

In [ ]:
if not df.empty:
    by_attack = df.groupby('attack_type')['complied'].apply(lambda s: s.astype(bool).mean()).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(9, max(4, len(by_attack) * 0.5)))
    colors = sns.color_palette('Reds_r', len(by_attack))
    by_attack.mul(100).sort_values().plot.barh(ax=ax, color=colors)
    ax.set_xlabel('Attack Success Rate (%)')
    ax.set_title('ASR by Attack Type (Baseline)')
    plt.tight_layout()
    plt.show()
    
    display(by_attack.mul(100).round(1).rename('ASR (%)').to_frame())

## 5. Heatmap: Category × Attack Type

In [ ]:
if not df.empty:
    pivot = df.groupby(['category', 'attack_type'])['complied'].apply(
        lambda s: s.astype(bool).mean()
    ).unstack(fill_value=0.0)
    
    if not pivot.empty and pivot.shape[1] > 1:
        fig, ax = plt.subplots(figsize=(max(8, pivot.shape[1]*1.5), max(5, pivot.shape[0]*0.7)))
        sns.heatmap(pivot * 100, annot=True, fmt='.0f', cmap='YlOrRd',
                    vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'ASR (%)'})
        ax.set_title('ASR (%) — Category × Attack Type')
        plt.tight_layout()
        plt.show()
    else:
        print('Need data with multiple attack types for heatmap.')

## 6. Inspect Complied Responses

In [ ]:
if not df.empty:
    complied = df[df['complied'].astype(bool)].copy()
    print(f'{len(complied)} complied responses.\n')
    
    # Show first 3 examples
    for _, row in complied.head(3).iterrows():
        print(f"ID: {row['id']}")
        print(f"Category: {row['category']}  |  Attack type: {row['attack_type']}")
        print(f"Goal: {str(row['goal'])[:100]}")
        print(f"Response: {str(row['response'])[:200]}")
        print('-' * 60)

## 7. Defense Comparison

In [ ]:
if not df.empty:
    baseline_asr = df['complied'].astype(bool).mean()
    rows = [{'defense': 'baseline', 'asr': baseline_asr, 'n': len(df)}]
    
    for name, path in DEFENSE_FILES.items():
        if path.exists():
            d = load_jsonl(path)
            rows.append({'defense': name, 'asr': d['complied'].astype(bool).mean(), 'n': len(d)})
    
    comp = pd.DataFrame(rows)
    comp['delta'] = comp['asr'] - baseline_asr
    comp['asr_pct'] = comp['asr'] * 100
    
    display(comp[['defense', 'n', 'asr_pct', 'delta']].round(3))
    
    if len(comp) > 1:
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ['#d62728' if r['defense'] == 'baseline' else '#2ca02c'
                  for _, r in comp.iterrows()]
        bars = ax.bar(comp['defense'], comp['asr_pct'], color=colors)
        ax.bar_label(bars, fmt='%.1f%%')
        ax.set_ylabel('ASR (%)')
        ax.set_title('Effect of Defenses on Attack Success Rate')
        plt.tight_layout()
        plt.show()

## 8. Run Full Analysis (generate figures to disk)

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '../src/analyze.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)